# 05 — Append-Only Filtering (FilteringPress)

This notebook demonstrates the **append-only filtering** compression
strategy: during generation, each new token is scored and either kept
or skipped before entering the cache. The cache only grows, never
shrinks — compression comes from skipping low-scoring tokens.

This mirrors kvpress's `FilteringPress`, which wraps a scoring press
(like `KeyDiffPress`) and applies a keep/skip threshold at each decode
step. Decisions are made **per head** independently — each head scores
all tokens and decides whether the new one survives. When a head
rejects a token, that position becomes padding for that head only,
creating ragged per-head lengths. The token is removed from the cache
only when **all heads** reject it.

We run three experiments:
1. **Filtering decisions** — simulate decode steps with per-head
   keep/skip decisions and ragged length tracking
2. **Cross-validation** — verify our decisions match kvpress's
   `FilteringPress` on the same key sequence
3. **Full pipeline on paged cache** — same filtering but with keys
   living in the Flash Attention paged cache

## Imports and Setup

In [ ]:
import sys

import torch
import torch.nn.functional as F
from vllm import _custom_ops as ops

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
sys.path.insert(0, "/opt/app-root/src/kvpress-fork")
from kvpress import KeyDiffPress, FilteringPress

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
PREFILL_LEN = 64
NUM_DECODE_STEPS = 32
COMPRESSION_RATIO = 0.5
DEVICE = "cuda"

## Primitives

Cache operations from notebook 02, scoring from notebook 03, plus
helpers for FilteringPress's per-head ragged length management:
`build_valid_mask`, `accept_last`, `fill_padding`, and `shrink`.

In [ ]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values


print("Cache functions defined")

In [ ]:
def keydiff_score(keys):
    """Score keys using KeyDiff's key-similarity metric.

    keys: [seq_len, num_kv_heads, head_dim]
    Returns: [num_kv_heads, seq_len]. Higher scores = more important.
    """
    keys_by_head = keys.permute(1, 0, 2)
    normalized = F.normalize(keys_by_head, p=2, dim=-1)
    anchor = normalized.mean(dim=1, keepdim=True)
    scores = -F.cosine_similarity(keys_by_head, anchor, dim=-1)
    return scores


def build_valid_mask(lengths, seq_len, device):
    """Per-head valid mask from ragged lengths, with the last position
    always valid (the new token being evaluated).

    lengths: [num_kv_heads]
    Returns: [num_kv_heads, seq_len]
    """
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)
    mask[:, -1] = True
    return mask


def accept_last(keys, lengths, accepted):
    """Move the last position into each accepting head's first padding
    slot, keeping valid data packed as a prefix.

    keys: [seq_len, num_kv_heads, head_dim] — modified in place
    lengths: [num_kv_heads] — modified in place
    accepted: [num_kv_heads] bool
    """
    last_pos = keys.shape[0] - 1
    for h in range(keys.shape[1]):
        if not accepted[h]:
            continue
        gap = lengths[h].item()
        if gap < last_pos:
            keys[gap, h] = keys[last_pos, h]
            keys[last_pos, h] = 0.0
        lengths[h] += 1


def fill_padding(keys, lengths):
    """Zero out padding positions per head."""
    for h in range(keys.shape[1]):
        L = lengths[h].item()
        if L < keys.shape[0]:
            keys[L:, h] = 0.0


def shrink(keys, lengths):
    """Remove trailing positions that are padding for all heads."""
    while keys.shape[0] > 0 and lengths.max().item() < keys.shape[0]:
        keys = keys[:-1]
    return keys


print("Scoring and ragged-length functions defined")

## Experiment 1 — Filtering Decisions

Simulate a sequence of decode steps. At each step:
1. Append the new token's key to the cache
2. Build a per-head valid mask (prefix per head + last position)
3. Score all keys with KeyDiff, masking invalid positions to `-inf`
4. Compute per-head rejection: a head rejects when the new token's
   score is strictly below the top-k threshold
5. Skip the token only when **all heads** reject it; otherwise accept
   per-head (rejecting heads get padding at that position)

In [ ]:
all_keys = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

print(f"Prefill tokens:     {PREFILL_LEN}")
print(f"Decode steps:       {NUM_DECODE_STEPS}")
print(f"Compression ratio:  {COMPRESSION_RATIO}")
print(f"Total keys:         {all_keys.shape[0]}")

In [ ]:
cached_keys = all_keys[:PREFILL_LEN].clone()
lengths = torch.full((NUM_KV_HEADS,), PREFILL_LEN, dtype=torch.long, device=DEVICE)
decisions = []

for step in range(NUM_DECODE_STEPS):
    new_key = all_keys[PREFILL_LEN + step]
    total_tokens_seen = PREFILL_LEN + step + 1
    cache_len_before = cached_keys.shape[0]

    keys_with_new = torch.cat([cached_keys, new_key.unsqueeze(0)], dim=0)
    seq_len = keys_with_new.shape[0]

    valid_mask = build_valid_mask(lengths, seq_len, DEVICE)

    scores = keydiff_score(keys_with_new)
    scores[~valid_mask] = float("-inf")

    n_kept = round(total_tokens_seen * (1 - COMPRESSION_RATIO))
    n_kept = max(1, min(n_kept, seq_len))
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    rejected = scores[:, -1] < threshold

    keep = not rejected.all().item()

    if keep:
        accepted = ~rejected
        accept_last(keys_with_new, lengths, accepted)
        fill_padding(keys_with_new, lengths)
        keys_with_new = shrink(keys_with_new, lengths)
        cached_keys = keys_with_new

    decisions.append({
        "step": step,
        "total_seen": total_tokens_seen,
        "cache_len_before": cache_len_before,
        "rejected_per_head": rejected.clone(),
        "keep": keep,
        "lengths_after": lengths.clone(),
    })

kept_count = sum(1 for d in decisions if d["keep"])
skipped_count = NUM_DECODE_STEPS - kept_count
effective_ratio = skipped_count / (PREFILL_LEN + NUM_DECODE_STEPS)

print(f"Results:")
print(f"  Kept:    {kept_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Skipped: {skipped_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Final cache size: {cached_keys.shape[0]} tokens")
print(f"  Per-head lengths: {lengths.tolist()}")
print(f"  Effective compression: {effective_ratio:.2%} of total tokens skipped")
print()

for d in decisions:
    n_rejected = d['rejected_per_head'].sum().item()
    marker = "KEEP" if d['keep'] else "SKIP"
    print(
        f"  step {d['step']:2d}  seen={d['total_seen']:3d}  "
        f"cache={d['cache_len_before']:3d}  "
        f"rejected={n_rejected}/{NUM_KV_HEADS}  {marker}"
    )

## Experiment 2 — Cross-Validation Against kvpress

Run the same key sequence through kvpress's `FilteringPress` and verify
that the per-head lengths match our standalone implementation after each
decode step.

This requires a mock `nn.Module` with a `layer_idx` attribute, and
`position_ids` in kwargs — the only external dependencies of
`FilteringPress.compress()`.

In [ ]:
from types import SimpleNamespace

mock_module = SimpleNamespace(layer_idx=0, head_dim=HEAD_SIZE)

fp = FilteringPress(
    base_press=KeyDiffPress(),
    target_compression_ratio=COMPRESSION_RATIO,
)

kvpress_keys = all_keys[:PREFILL_LEN].permute(1, 0, 2).unsqueeze(0).clone()
kvpress_values = torch.zeros_like(kvpress_keys)

mismatches = 0
fp.reset()

for step in range(NUM_DECODE_STEPS):
    new_key = all_keys[PREFILL_LEN + step]
    total_tokens_seen = PREFILL_LEN + step + 1

    new_key_kvpress = new_key.unsqueeze(0).unsqueeze(0).permute(0, 2, 1, 3)
    keys_in = torch.cat([kvpress_keys, new_key_kvpress], dim=2)
    values_in = torch.cat(
        [kvpress_values, torch.zeros_like(new_key_kvpress)], dim=2,
    )

    position_ids = torch.arange(total_tokens_seen, device=DEVICE).unsqueeze(0)

    keys_out, values_out = fp.compress(
        module=mock_module,
        hidden_states=None,
        keys=keys_in,
        values=values_in,
        attentions=None,
        kwargs={"position_ids": position_ids},
    )

    if 0 in fp._lengths:
        kvpress_lengths = fp._lengths[0][0]
    else:
        kvpress_lengths = torch.full(
            (NUM_KV_HEADS,), PREFILL_LEN, dtype=torch.long, device=DEVICE,
        )

    our_lengths = decisions[step]["lengths_after"]

    if not torch.equal(our_lengths, kvpress_lengths):
        mismatches += 1
        print(
            f"  MISMATCH step {step}: "
            f"ours={our_lengths.tolist()}, kvpress={kvpress_lengths.tolist()}"
        )

    kvpress_keys = keys_out
    kvpress_values = values_out

if mismatches == 0:
    print(
        f"All {NUM_DECODE_STEPS} filtering decisions match "
        f"kvpress FilteringPress (per-head lengths identical)"
    )
else:
    print(f"\n{mismatches}/{NUM_DECODE_STEPS} decisions differ")

## Experiment 3 — Full Pipeline on Paged Cache

End-to-end: keys live in the Flash Attention paged cache. At each decode
step, write the new token to the cache, gather all keys, score with the
valid mask, and apply per-head filtering with gap-filling — all operating
directly on paged cache slots.

In [ ]:
def accept_last_paged(key_cache, value_cache, block_table, block_size,
                      new_pos, lengths, accepted, device):
    """Move the new token from new_pos to each accepting head's first
    gap in the paged cache."""
    new_slot = build_slot_mapping_for_positions(
        block_table, torch.tensor([new_pos], device=device), block_size,
    )
    new_block = (new_slot // block_size).item()
    new_offset = (new_slot % block_size).item()

    for h in range(len(lengths)):
        if not accepted[h]:
            continue
        gap = lengths[h].item()
        if gap < new_pos:
            gap_slot = build_slot_mapping_for_positions(
                block_table, torch.tensor([gap], device=device), block_size,
            )
            gap_block = (gap_slot // block_size).item()
            gap_offset = (gap_slot % block_size).item()
            key_cache[gap_block, gap_offset, h] = key_cache[new_block, new_offset, h]
            value_cache[gap_block, gap_offset, h] = value_cache[new_block, new_offset, h]
            key_cache[new_block, new_offset, h] = 0.0
            value_cache[new_block, new_offset, h] = 0.0
        lengths[h] += 1


def fill_padding_paged(key_cache, value_cache, block_table, block_size,
                       max_pos, lengths, device):
    """Zero out padding positions per head in the paged cache."""
    for h in range(len(lengths)):
        L = lengths[h].item()
        if L >= max_pos:
            continue
        padding_positions = torch.arange(L, max_pos, dtype=torch.long, device=device)
        padding_slots = build_slot_mapping_for_positions(
            block_table, padding_positions, block_size,
        )
        blocks = padding_slots // block_size
        offsets = padding_slots % block_size
        key_cache[blocks, offsets, h] = 0.0
        value_cache[blocks, offsets, h] = 0.0


all_values = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

total_len = PREFILL_LEN + NUM_DECODE_STEPS
num_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

num_seq_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)
k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

prefill_positions = torch.arange(PREFILL_LEN, dtype=torch.long, device=DEVICE)
prefill_slots = build_slot_mapping_for_positions(
    block_table, prefill_positions, BLOCK_SIZE,
)
ops.reshape_and_cache_flash(
    all_keys[:PREFILL_LEN], all_values[:PREFILL_LEN],
    key_cache, value_cache,
    prefill_slots, "auto", k_scale, v_scale,
)

print("Paged-cache filtering helpers defined")
print(f"Prefilled {PREFILL_LEN} tokens into paged cache")

In [ ]:
max_cache_pos = PREFILL_LEN
paged_lengths = torch.full((NUM_KV_HEADS,), PREFILL_LEN, dtype=torch.long, device=DEVICE)
paged_decisions = []

for step in range(NUM_DECODE_STEPS):
    total_tokens_seen = PREFILL_LEN + step + 1
    new_key = all_keys[PREFILL_LEN + step].unsqueeze(0)
    new_value = all_values[PREFILL_LEN + step].unsqueeze(0)

    new_position = torch.tensor([max_cache_pos], dtype=torch.long, device=DEVICE)
    new_slot = build_slot_mapping_for_positions(
        block_table, new_position, BLOCK_SIZE,
    )
    ops.reshape_and_cache_flash(
        new_key, new_value,
        key_cache, value_cache,
        new_slot, "auto", k_scale, v_scale,
    )

    all_positions = torch.arange(max_cache_pos + 1, dtype=torch.long, device=DEVICE)
    all_slots = build_slot_mapping_for_positions(
        block_table, all_positions, BLOCK_SIZE,
    )
    cached_keys_paged, _ = gather_from_paged_cache(
        key_cache, value_cache, all_slots, BLOCK_SIZE,
    )

    seq_len = cached_keys_paged.shape[0]
    valid_mask = build_valid_mask(paged_lengths, seq_len, DEVICE)
    scores = keydiff_score(cached_keys_paged)
    scores[~valid_mask] = float("-inf")

    n_kept = max(1, int(total_tokens_seen * (1 - COMPRESSION_RATIO)))
    n_kept = min(n_kept, seq_len)
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    rejected = scores[:, -1] < threshold

    keep = not rejected.all().item()
    paged_decisions.append(keep)

    if not keep:
        block_idx = (new_slot // BLOCK_SIZE).item()
        offset = (new_slot % BLOCK_SIZE).item()
        key_cache[block_idx, offset] = 0.0
        value_cache[block_idx, offset] = 0.0
        continue

    accepted = ~rejected
    accept_last_paged(
        key_cache, value_cache, block_table, BLOCK_SIZE,
        max_cache_pos, paged_lengths, accepted, DEVICE,
    )
    fill_padding_paged(
        key_cache, value_cache, block_table, BLOCK_SIZE,
        max_cache_pos + 1, paged_lengths, DEVICE,
    )

    if paged_lengths.max().item() >= max_cache_pos + 1:
        max_cache_pos += 1

dense_decisions = [d["keep"] for d in decisions]

mismatches = sum(
    1 for p, d in zip(paged_decisions, dense_decisions) if p != d
)

print(f"Final max cache position: {max_cache_pos}")
print(f"Per-head lengths: {paged_lengths.tolist()}")
print(f"Kept: {sum(paged_decisions)}/{NUM_DECODE_STEPS} decode tokens")
print()

if mismatches == 0:
    print(
        f"All {NUM_DECODE_STEPS} decisions match between "
        f"paged-cache and dense-tensor pipelines"
    )
else:
    print(f"{mismatches}/{NUM_DECODE_STEPS} decisions differ")
    for i, (p, d) in enumerate(zip(paged_decisions, dense_decisions)):
        if p != d:
            print(f"  step {i}: paged={p}, dense={d}")

## Notes and Next Steps

**FilteringPress emulation validated.** The standalone per-head filtering
logic makes the same keep/skip decisions as kvpress's `FilteringPress`,
including ragged per-head lengths, accept-last gap-filling, and padding
management. These decisions are identical whether operating on dense
tensors or on gathered paged cache data.

**What this enables:** With scoring and per-head filtering validated on
the Flash Attention cache layout, the next step is integrating this into
vLLM's `Attention.forward()`. The integration will:
1. Gather cached keys after `do_kv_cache_update`
2. Score with `keydiff_score`
3. Apply per-head filtering with ragged length tracking
4. Manage gap-filling and padding in the paged cache
5. Track logical positions for RoPE separately from cache positions